# Flow variation in a pump recording

**FORGE | Exploratory analysis of SKAB experiment `valve1/1`**

This analysis examines whether flow measurements distinguish observations labeled normal from those labeled anomalous in a laboratory pump recording.

Anomalous observations had a lower median flow, but the two groups shared part of their observed ranges. A trailing rolling median revealed the timing of sustained variation. These findings describe one development recording; they do not measure anomaly-detector performance.


## Dataset and provenance

The recording comes from the [Skoltech Anomaly Benchmark (SKAB)](https://github.com/waico/SKAB/blob/b2c0d46c2971dcbfe71e26087b6d231998bb91c2/README.md), developed by Iurii D. Katser and Vyacheslav O. Kozitsin. Experiment `valve1/1` belongs to the inlet-valve closure experiments in the [source catalog](https://github.com/waico/SKAB/blob/b2c0d46c2971dcbfe71e26087b6d231998bb91c2/data/README.md).

The sample contains 1,145 observations across 11 columns and spans 20 minutes. The [sample manifest](../data/sample-manifest.json) records its source revision, checksum, and local location. The dataset retains its source license, documented in the [data notes](../data/README.md).

The loader verifies the checksum and validates required columns, timestamp order, numeric values, and annotations. This recording is reserved for development and excluded from the future final test set because its annotations have been inspected.


In [ ]:
from forge.data.sample import load_sample

df = load_sample()
print(f"Loaded sample recording: {df.shape}")

In [ ]:
print(df.head(5))
column_names = df.columns.to_list()
print(column_names)

## Measurements and reference annotations

The recording contains eight sensor channels, a timestamp, and two source annotation columns.

| Columns | Role |
|---|---|
| `Accelerometer1RMS`, `Accelerometer2RMS`, `Current`, `Pressure`, `Temperature`, `Thermocouple`, `Voltage`, `Volume Flow RateRMS` | Sensor measurements |
| `datetime` | Observation time |
| `anomaly` | Source label: 0 for normal observations, 1 for anomalous observations |
| `changepoint` | Source marker: 1 at a marked transition, 0 elsewhere |

These definitions follow the [SKAB documentation](https://github.com/waico/SKAB/blob/b2c0d46c2971dcbfe71e26087b6d231998bb91c2/README.md#datasets). An anomalous period can span many observations; a change point marks a transition, including a return to the normal label.

Source annotations provide reference labels for this comparison and are excluded from the sensor input table.


In [ ]:
# extend dataframe
sensor_columns = df.columns.difference(["datetime", "anomaly", "changepoint"])
label_columns = df.columns.intersection(["anomaly", "changepoint"])
print(f"Sensor columns: {sensor_columns.to_list()}")
print(f"Label columns: {label_columns.to_list()}")

In [ ]:
print(df["anomaly"].value_counts())

The sensor table `X` contains eight columns, and `labels` contains the two annotation columns. Both preserve row alignment with the original recording.

Column extraction uses the pinned sample's order: the timestamp appears first and the annotations appear last. This assumption limits reuse of the current extraction code with other schemas.


In [ ]:
X=df[column_names[1:len(column_names)-2]]
labels = df[column_names[len(column_names)-2: len(column_names)]]
print(X)
print(labels)
print(f"X shape: {X.shape} | X column names: {X.columns.to_list()}")
print(f"Labels shape: {labels.shape} | Labels column names: {labels.columns.to_list()}")



## Flow comparison

Flow from `Volume Flow RateRMS`, measured in liters per minute (L/min), was grouped by the source `anomaly` label. Each group was summarized by its count, median, minimum, and maximum.


In [ ]:
# Select normal_flow and anomalous_flow from df.
normal_flow = df.loc[df["anomaly"]==0, "Volume Flow RateRMS"]
anomalous_flow = df.loc[df["anomaly"]==1, "Volume Flow RateRMS"]


print("Normal flow statistics:")
print(f"Count: {normal_flow.count()}")
print(f"Median: {normal_flow.median()}")
print(f"Min: {normal_flow.min()}")
print(f"Max: {normal_flow.max()}")

print("Anomalous flow statistics:")
print(f"Count: {anomalous_flow.count()}")
print(f"Median: {anomalous_flow.median()}")
print(f"Min: {anomalous_flow.min()}")
print(f"Max: {anomalous_flow.max()}")


### Lower median flow with overlapping ranges

| Source label | Observations | Median flow (L/min) | Observed range (L/min) |
|---|---:|---:|---:|
| Normal (`anomaly = 0`) | 743 | 32 | 31–33 |
| Anomalous (`anomaly = 1`) | 402 | 31 | 30–32 |

The anomalous group had a lower median, while the observed ranges overlapped between 31 and 32 L/min. A single instantaneous flow cutoff therefore cannot perfectly separate the source labels in this recording.


## Temporal variation

Flow measurements were indexed by `datetime` and summarized using a trailing 30-second rolling median. Each value uses the current observation and earlier observations within the window; initial windows contain only the available observations.

The recording contains 56 two-second sampling intervals among otherwise one-second intervals. A time-based window follows elapsed time without inserting measurements into these gaps.


In [ ]:
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

flow_by_time = df.set_index("datetime")["Volume Flow RateRMS"]
print(flow_by_time.head())
# Median of available observations in the preceding 30 seconds.
flow_trend = flow_by_time.rolling("30s").median()
print(flow_trend.head(10))

In [ ]:
fig = px.scatter(df, x="datetime", y="Volume Flow RateRMS", color="anomaly", title="Volume Flow RateRMS Over Time")
fig.add_scatter(x=flow_trend.index, y=flow_trend, mode="lines", name="30s Rolling Median", line=dict(color="black"))

**Flow visualization.** Running the preceding cells generates a scatter plot of measured flow, colored by source anomaly labels, with the trailing median in black. Saved chart outputs are omitted from the repository; the numerical summary is retained above.

The smoothed series showed a sustained reduction around 10:45 and an increase around 10:51. This temporal view locates variation that the grouped statistics summarize across the recording.


## Interpretation and limitations

Flow alone does not fully separate the source labels, motivating evaluation of temporal and multivariate features. The comparison pools normal observations before and after the anomalous period, which may conceal differences between those periods.

The rolling median emphasizes sustained variation but can suppress brief changes and respond later than individual measurements. This behavior is not a measured detection delay because no detector or alert rule was evaluated.

These results describe one laboratory experiment and do not establish a failure mechanism or performance across equipment and operating conditions. Subsequent evaluation requires separate experiments for training, validation, and testing, with transformations and thresholds fitted without access to the test set.
